In [6]:
import pandas as pd
import glob
import os

# 1. 파일 로드 함수 (데이터 타입 유지)
def load_csv(path):
    for enc in ['cp949', 'utf-8', 'utf-8-sig']:
        try:
            # 업체코드는 무조건 문자열 6자리로 로드
            df = pd.read_csv(path, encoding=enc, dtype={'업체코드': str})
            df.columns = df.columns.str.strip()
            # Unnamed 컬럼 제거
            df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
            return df
        except:
            continue
    return None

# 2. 마스터 데이터(틀) 생성
print("단계 1: 마스터 데이터 생성 중...")
df_master = load_csv("분석대상_정상.csv")
# CL1, CL2 제외 로직 (정상 기업 추출)
clean = df_master[(df_master["CL1"] != "Y") & (df_master["CL2"] != "Y")].copy()

# 연도 틀 (2014~2024)
years_list = list(range(2014, 2025))
year_df = pd.DataFrame({'연도': years_list})

# 패널 기본 틀: 기업 x 연도
final_df = clean.assign(key=1).merge(year_df.assign(key=1), on='key').drop('key', axis=1)
final_df['업체코드'] = final_df['업체코드'].str.zfill(6)
final_df['연도'] = final_df['연도'].astype(int)

# 3. 모든 CSV 파일 순회하며 컬럼 추가
print("단계 2: 변수 컬럼 추가 시작...")
# ??_*.csv 패턴으로 모든 파일을 가져옵니다.
file_list = sorted(glob.glob("??_*.csv"))

for file in file_list:
    filename = os.path.basename(file)
    file_num = filename[:2]  # 파일명 앞 숫자 2자리 (01, 02...)
    
    # 변수명 정리 (파일명에서 숫자와 확장자 제거)
    # 예: "01_총자본증가율_통합.csv" -> "01_총자본증가율"
    var_name = filename.replace(".csv", "").replace("_통합", "").replace("(개별)", "")
    
    # 데이터 로드
    temp = load_csv(file)
    if temp is None: continue
    
    print(f"🔗 변수 삽입 중: {var_name}")
    
    # '업체코드' 전처리
    temp['업체코드'] = temp['업체코드'].astype(str).str.zfill(6)
    
    # 연도 컬럼 추출 (_2014 형식)
    year_cols = [c for c in temp.columns if c.startswith("_20")]
    
    # [핵심] 와이드 -> 롱 변환
    melted = temp[['업체코드'] + year_cols].melt(
        id_vars=['업체코드'], 
        var_name='연도', 
        value_name=var_name  # 이 파일의 변수명이 컬럼명이 됨
    )
    
    # 병합용 연도 형식 통일
    melted['연도'] = melted['연도'].str.replace('_', '').astype(int)
    
    # [병합] 기존 틀(final_df)에 '업체코드'와 '연도'를 기준으로 새 변수 컬럼을 붙임
    final_df = pd.merge(
        final_df, 
        melted[['업체코드', '연도', var_name]], 
        on=['업체코드', '연도'], 
        how='left'
    )

# 4. 결과 저장
print("단계 3: 최종 파일 저장 중...")
final_df.drop(columns=['CL1', 'CL2'], errors='ignore', inplace=True)

# 저장 전 최종 확인: 컬럼에 변수 숫자들이 잘 들어갔는지
print(f"최종 생성된 컬럼 수: {len(final_df.columns)}")
final_df.to_csv("dataset_정상.csv", index=False, encoding='utf-8-sig')

print("✨ 모든 변수 값이 성공적으로 통합되었습니다!")

단계 1: 마스터 데이터 생성 중...
단계 2: 변수 컬럼 추가 시작...
🔗 변수 삽입 중: 01_총자본증가율
🔗 변수 삽입 중: 02_영업이익증가율
🔗 변수 삽입 중: 03_순이익증가율
🔗 변수 삽입 중: 04_자기자본증가율
🔗 변수 삽입 중: 05_매출액증가율
🔗 변수 삽입 중: 06_종업원수증가율
🔗 변수 삽입 중: 07_매출액총이익율
🔗 변수 삽입 중: 08_매출액영업이익율
🔗 변수 삽입 중: 09_매출액순이익율
🔗 변수 삽입 중: 10_총자산영업이익율
🔗 변수 삽입 중: 11_총자산순이익율
🔗 변수 삽입 중: 12_자기자본영업이익율
🔗 변수 삽입 중: 13_자기자본순이익율
🔗 변수 삽입 중: 14_금융비용부담율
🔗 변수 삽입 중: 15_수지비율
🔗 변수 삽입 중: 16_사내유보대자기자본비율
🔗 변수 삽입 중: 17_총자본회전율
🔗 변수 삽입 중: 18_자기자본회전율
🔗 변수 삽입 중: 19_타인자본회전율
🔗 변수 삽입 중: 20_유동자산회전율
🔗 변수 삽입 중: 21_당좌자산회전율
🔗 변수 삽입 중: 22_재고자산회전율
🔗 변수 삽입 중: 23_매출채권회전율
🔗 변수 삽입 중: 24_순운전자본회전율
🔗 변수 삽입 중: 26_종업원1인당부가가치
🔗 변수 삽입 중: 27_노동장비율
🔗 변수 삽입 중: 28_기계장비율
🔗 변수 삽입 중: 29_자본집약도
🔗 변수 삽입 중: 30_총자본투자효율
🔗 변수 삽입 중: 31_설비투자효율
🔗 변수 삽입 중: 32_1주당매출액
🔗 변수 삽입 중: 33_주당순이익
🔗 변수 삽입 중: 34_주당현금흐름
🔗 변수 삽입 중: 35_주당순자산가치
🔗 변수 삽입 중: 36_유보율
🔗 변수 삽입 중: 37_자기자본구성비율
🔗 변수 삽입 중: 38_유동비율
🔗 변수 삽입 중: 39_당좌비율
🔗 변수 삽입 중: 40_현금비율
🔗 변수 삽입 중: 41_재고자산대순운전자본비율
🔗 변수 삽입 중: 42_매출채권대매입채무비율
🔗 변수 삽입 중: 43_부채비율
🔗 변수 삽입 중: 44_이자보상비율
🔗 변수 삽입 중: 45_Cashflow대